# Landing → Bronze: medidores

**Fonte:** `medidores.csv` (parque de medidores, com instalação, retirada e manutenção), separador vírgula, com cabeçalho.
**Destino:** `venergy.bronze.medidores` (Delta).

**Decisões desta camada**
- Conteúdo preservado: schema explícito com todas as colunas como texto (`STRING`), equivalente a `inferSchema = false`. Sem `trim`, limpeza, deduplicação ou conversão. O tratamento é feito na Silver.
- Contrato do cabeçalho: com `enforceSchema = false`, o Spark compara o cabeçalho do arquivo com `EXPECTED_COLUMNS`. Se a origem renomear ou reordenar colunas, a carga falha em vez de gravar dados deslocados.
- Linhas malformadas: `mode = FAILFAST`. Uma linha com número de colunas diferente do contrato interrompe a carga em vez de ser descartada ou completada com `null`.
- Colunas de rastreio: `source_file` (arquivo de origem) e `ingestion_datetime` (momento da carga), padrão do time.
- Carga completa com `overwrite`: o dataset é uma extração única, então reexecutar o notebook não duplica registros (o Workflow precisa ser reexecutável de ponta a ponta).
- Observação: o leitor CSV do Spark transforma campo vazio em `null`. Em `data_retirada` e `data_ultima_manutencao` isso vale para os vazios; a sentinela `9999-12-31` chega como texto e é documentada.

As falhas encontradas estão em `docs/auditoria_dados/auditoria_medidores.md`.

In [ ]:
catalog = "venergy"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"
volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

medidores_path = f"{volume_path}/dados/medidores.csv"
bronze_table = f"{catalog}.{bronze_schema}.medidores"

EXPECTED_COLUMNS = [
    "id_medidor",
    "id_uc",
    "modelo",
    "tipo",
    "data_instalacao",
    "data_retirada",
    "data_ultima_manutencao",
    "status",
]
EXPECTED_ROWS = 51_812

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

schema_bruto = StructType([StructField(c, StringType(), True) for c in EXPECTED_COLUMNS])

df_medidores = (
    spark.read
    .schema(schema_bruto)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("ignoreLeadingWhiteSpace", False)
    .option("ignoreTrailingWhiteSpace", False)
    .csv(medidores_path)
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_datetime", F.current_timestamp())
)

In [ ]:
(
    df_medidores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

In [ ]:
# Contagem lida como texto puro (sem parser CSV), descontando o cabeçalho
linhas_arquivo = spark.read.text(medidores_path).filter(F.length("value") > 0).count() - 1
tabela = spark.table(bronze_table)
linhas_tabela = tabela.count()

print({"linhas_arquivo": linhas_arquivo, "linhas_tabela": linhas_tabela, "esperado": EXPECTED_ROWS})
assert tabela.columns == EXPECTED_COLUMNS + ["source_file", "ingestion_datetime"], f"Colunas inesperadas: {tabela.columns}"
assert linhas_tabela == linhas_arquivo, "Contagem da tabela diverge do arquivo bruto"
assert linhas_tabela == EXPECTED_ROWS, "Contagem diverge do volume esperado da extração"
print("OK: 51.812 linhas carregadas em", bronze_table)

## Evidências das falhas (consultas sobre a Bronze)
As consultas abaixo reproduzem os principais achados do relatório de auditoria. Elas não alteram dados.

In [ ]:
t = bronze_table
cols = ", ".join(EXPECTED_COLUMNS)

# 1. Duplicatas exatas (linha inteira repetida, inclusive id_medidor)
display(spark.sql(f"""
  SELECT COUNT(*) AS linhas,
         COUNT(DISTINCT id_medidor) AS ids_distintos,
         COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT {cols} FROM {t})) AS duplicatas_exatas
  FROM {t}"""))

# 2. Grafias de tipo
display(spark.sql(f"SELECT tipo, COUNT(*) AS linhas FROM {t} GROUP BY 1 ORDER BY 2 DESC"))

# 3. Espaços nas extremidades de modelo
display(spark.sql(f"""
  SELECT COUNT_IF(modelo <> trim(modelo)) AS com_espacos,
         COUNT(DISTINCT modelo) AS variacoes_brutas,
         COUNT(DISTINCT trim(modelo)) AS modelos_reais
  FROM {t}"""))

# 4. Formatos de data_instalacao (troca cada dígito por 9 e conta os padrões)
display(spark.sql(f"""
  SELECT regexp_replace(data_instalacao, '[0-9]', '9') AS formato, COUNT(*) AS linhas
  FROM {t} GROUP BY 1 ORDER BY 2 DESC"""))

# 5. Prova da ordem dia/mês por separador (campo > 12 só pode ser dia)
display(spark.sql(f"""
  SELECT substr(data_instalacao, 3, 1) AS separador, COUNT(*) AS linhas,
         COUNT_IF(try_cast(substr(data_instalacao, 1, 2) AS INT) > 12) AS primeiro_campo_maior_12,
         COUNT_IF(try_cast(substr(data_instalacao, 4, 2) AS INT) > 12) AS segundo_campo_maior_12
  FROM {t} WHERE data_instalacao RLIKE '^[0-9]{{2}}[-/.]' GROUP BY 1 ORDER BY 1"""))

# 6. data_retirada x status (vazio e sentinela 9999-12-31)
display(spark.sql(f"""
  SELECT status,
         CASE WHEN data_retirada IS NULL THEN 'vazio'
              WHEN data_retirada = '9999-12-31' THEN '9999-12-31'
              ELSE 'data real' END AS data_retirada,
         COUNT(*) AS linhas
  FROM {t} GROUP BY 1, 2 ORDER BY 1, 2"""))

# 7. UCs com mais de um medidor ATIVO, antes e depois de remover as duplicatas exatas
display(spark.sql(f"""
  SELECT 'bruto' AS base, COUNT(*) AS ucs_com_mais_de_um_ativo
  FROM (SELECT id_uc FROM {t} WHERE status = 'ATIVO' GROUP BY id_uc HAVING COUNT(*) > 1)
  UNION ALL
  SELECT 'sem duplicatas exatas', COUNT(*)
  FROM (SELECT id_uc FROM (SELECT DISTINCT {cols} FROM {t}) WHERE status = 'ATIVO' GROUP BY id_uc HAVING COUNT(*) > 1)"""))

# 8. Trocas de medidor por mês de retirada
display(spark.sql(f"""
  SELECT substr(data_retirada, 1, 7) AS mes_retirada, COUNT(DISTINCT id_medidor) AS medidores_retirados
  FROM {t} WHERE status = 'RETIRADO' GROUP BY 1 ORDER BY 1"""))